
## Explore data of news and tweets .parquet files
In this step we need install all requeriments to extract and convert .parquet files in dataframes

In [1]:
pip install pandas pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import json
import ast

## Dataframe that normalizes the information in the news .parquet files

In [3]:
df = pd.read_parquet("../datalake_silver/noticias_processed_20260530_012014.parquet")
print(df.columns)
# 1. Convertir comments de string a lista
df["comments"] = df["comments"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# 2. Expandir: una fila por comentario, manteniendo el link asociado
df_expandido = df.explode("comments").reset_index(drop=True)

# 3. Quedarte solo con las columnas relevantes
df_expandido = df_expandido[["newsLink", "comments"]]

# Opción 3: La más bonita, con truncado controlado
df_expandido["newsLink_corto"] = df_expandido["newsLink"].str.split("/").str[-1].str[:40]
df_expandido["comentario_corto"] = df_expandido["comments"].str[:80]
print("Date:", df)
print(df_expandido[["newsLink_corto", "comentario_corto"]].to_string())

Index(['newsLink', 'comments', '_id', 'snapshot_id', 'snapshot_date'], dtype='str')
Date:                                             newsLink  \
0  https://www.gsmarena.com/xiaomi_unveils_smart_...   
1  https://www.gsmarena.com/samsung_is_now_shippi...   
2  https://www.gsmarena.com/samsung_galaxy_z_fold...   
3  https://www.gsmarena.com/vertu_unveils_alphafo...   
4  https://www.gsmarena.com/hmd_grand_renders_and...   
5  https://www.gsmarena.com/oura_ring_5_announced...   
6  https://www.arenaev.com/xiaomi_ceo_personally_...   
7  https://www.gsmarena.com/honor_x80s_launch_dat...   

                                            comments  \
0  [I like how at least some still make earbuds w...   
1  [Ai sux, Ai will ruin this planet, AI will hal...   
2  [Waiting for the day Samsung increases battery...   
3  [Rebranded zte with leather slapped onto it. D...   
4  [-, 28 May 20261. How did you get banned in un...   
5  [pl2rts, 7 hours agoYou get a really good midd...   
6  [Anonymous

## Dataframe that normalizes the information in the tweets .parquet files

In [4]:
df = pd.read_parquet("../datalake_silver/tweets_processed_20260530_012013.parquet")
print(df.columns)

# Relevant columns
TWEET_COLS = [
    "type", "id", "url", "twitterUrl", "text", "source",
    "retweetCount", "replyCount", "likeCount", "quoteCount", "viewCount",
    "createdAt", "lang", "isReply", "conversationId",
    "inReplyToId", "inReplyToUsername", "snapshot_date",
    "author_id", "author_userName", "author_name",
    "author_followers", "author_following",
    "author_isVerified", "author_isBlueVerified",
    "author_location", "author_description"
]


cols_existentes = [c for c in TWEET_COLS if c in df.columns]
df_completo = df[cols_existentes]

# Ver primera fila correctamente
print(df_completo["text"].count())

# Ver como tabla
print(df_completo.head(10))


Index(['type', 'id', 'url', 'twitterUrl', 'text', 'source', 'retweetCount',
       'replyCount', 'likeCount', 'quoteCount',
       ...
       'quoted_tweet_quoted_tweet_retweeted_tweet',
       'quoted_tweet_quoted_tweet_isLimitedReply',
       'quoted_tweet_quoted_tweet_communityInfo',
       'quoted_tweet_quoted_tweet_article',
       'author_profile_bio_entities_description_symbols',
       'entities_timestamps',
       'quoted_tweet_author_profile_bio_entities_description_hashtags',
       'quoted_tweet_author_profile_bio_entities_url_urls', 'text_cleaned',
       'text_length'],
      dtype='str', length=173)
60
    type                   id  \
0  tweet  2060004476669374743   
1  tweet  2059950505447584179   
2  tweet  2059950392176234910   
3  tweet  2059937735977398654   
4  tweet  2059914754030776460   
5  tweet  2059899870865932490   
6  tweet  2059872425068704029   
7  tweet  2059782299370070471   
8  tweet  2059696184248401937   
9  tweet  2059641565706871191   

           